robimy wszystkie importy i tworzymy root do zadań

In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import networkx as nx
import matplotlib.pyplot as plt
import csv
import random
from bs4 import XMLParsedAsHTMLWarning
import warnings
import shutil
import copy
from lxml import etree
from datetime import datetime, timedelta
import pytest
import ipytest
import os
from threading import Thread

warnings.filterwarnings("ignore", category=pytest.PytestAssertRewriteWarning)

In [2]:
def odczyt(nazwa_pliku):
    try:
        tree = etree.parse(nazwa_pliku)
        root = tree.getroot()
        if tree is not None:
            for elem in root.getiterator():
                elem.tag = etree.QName(elem).localname  # usuwamy namespaces z tagow, aby nie musiec dac tego w forach
            return root
    except:
        return None


root = odczyt('drugbank_partial.xml')

Zadanie1

In [3]:
def zadanie1(root):
    leki = None
    try:
        leki = root.findall('drug')
    except:
        return None
    elementy_ramki = []
    
    for l in leki:
        id = l.find("drugbank-id[@primary='true']").text
        nazwa = l.find('name').text
        typ = l.get("type")
        opis = l.find('description').text

        dozowanie = l.find('dosages')
        formy = {x.find('form').text for x in dozowanie.findall('dosage')}
        if len(formy) == 0:
            formy = {}

        wskazania = l.find('indication').text
        mechanizm_dzialania = l.find('mechanism-of-action').text

        jedzenie = l.find('food-interactions')
        interakcje_pokarmowe = {x.text for x in jedzenie.findall('food-interaction')}
        if len(interakcje_pokarmowe) == 0:
            interakcje_pokarmowe = {}

        el_ramki = {
            "id": id,
            "nazwa": nazwa,
            "typ": typ,
            "opis": opis,
            "formy_wystepowania": formy,
            "wskazania": wskazania,
            "mechanizm_dzialania": mechanizm_dzialania,
            "interakcje_z_jedzeniem": interakcje_pokarmowe
        }
        elementy_ramki.append(el_ramki)

    return pd.DataFrame(elementy_ramki)

ramka = zadanie1(root)
ramka.to_csv('zadanie1.txt', index = False)

Zadanie2


In [ ]:
def zadanie2przet(root):
    leki = None
    try:
        leki = root.findall('drug')
    except:
        return None
    ramka = []

    for l in leki:
        id = l.find("drugbank-id[@primary='true']").text
        synonimy = [s.text for s in l.find('synonyms').findall('synonym')]

        ramka.append({
            'id': id,
            'synonimy': synonimy
        })

    return pd.DataFrame(ramka)

ramka = zadanie2przet(root)

def zadanie2(drugbank_id, ramka):
	if not (ramka['id'] == drugbank_id).any():
		print("Zadanie 2 - dany lek nie istnieje - id = ", drugbank_id, end = '\n')
		return

	G = nx.Graph()
	G.add_node(drugbank_id)
	for s in ramka.loc[ramka['id'] == drugbank_id, 'synonimy'].values[0]:
		G.add_edge(drugbank_id, s)

	plt.figure(figsize=(15,5))
	plt.tight_layout()
	nx.draw(G, with_labels = True, node_size = 1000, font_size = 10)
	plt.show()
	G.clear()

ramka.to_csv('zadanie2.txt', index = False)
zadanie2('DB00001', ramka)

Zadanie3

In [5]:
def zadanie3(root):
    try:
        leki = root.findall('drug')
    except:
        return None
    ramka = []
    
    for l in leki:
        id = l.find("drugbank-id[@primary='true']").text
        for p in (l.find('products')).findall('product'):
            ramka.append({
                'id': id,
                'nazwa_produktu': p.find('name').text,
                'kod_ndc': p.find('ndc-product-code').text if p.find('ndc-product-code') is not None else None,
                'postac_wystepowania': p.find('dosage-form').text,
                'sposob_aplikacji': p.find('route').text,
                'informacja_o_dawce': p.find('strength').text,
                'kraj': p.find('country').text,
                'agencja': p.find('labeller').text
            })
    
    return pd.DataFrame(ramka)
ramka = zadanie3(root)
ramka.to_csv("zadanie3.txt", index = False)

Zadanie4

In [ ]:
def zadanie4(root):
    try:
        grupy_sciezek = root.findall('.//pathways')
    except:
        return -1

    # Nie chcemy brać powtórek, dlatego używamy dictionary, a nie zwykłej listy.
    sciezki = set()

    for gs in grupy_sciezek:
        for g in gs.findall('pathway'):
            leki = (g.find('drugs')).findall('drug')
            if len(leki) > 0:
                sciezki.add(g.find('smpdb-id').text)
    return len(sciezki)


print("Zadanie4 - odpowiedź: ", zadanie4(root), end = '\n')

Zadanie5

In [ ]:
def zadanie5przet(root):
    try:
        grupy_sciezek = root.findall('.//pathways')
    except:
        return None, None, None, None
    sciezki = {}
    nazwy = {}

    for gs in grupy_sciezek:
        for g in gs.findall('pathway'):
            klucz = g.find('smpdb-id').text
            nazwa = g.find('name').text
            nazwy[klucz] = nazwa
            if sciezki.get(klucz) is None:
                sciezki[klucz] = set()
            for l in g.find('drugs').findall('drug'):
                sciezki[klucz].add((l.find('drugbank-id').text, l.find('name').text))

    ramka = []
    G = nx.Graph()
    lstrona = set()
    pstrona = set()

    for x in sciezki.keys():
        for l in sciezki[x]:
            ramka.append({
                'id_sciezki': x,
                'nazwa_sciezki': nazwy[x],
                'id_leku': l[0],
                'nazwa_leku': l[1]
            })
            G.add_edge(nazwy[x], l[1])
            lstrona.add(nazwy[x])
            pstrona.add(l[1])

    ramka = pd.DataFrame(ramka)
    polozenie = {}
    kolory = []

    for i, l in enumerate(lstrona):
        polozenie[l] = (-1, i)
    for i, p in enumerate(pstrona):
        polozenie[p] = (1, i)

    for x in G.nodes():
        if polozenie[x][0] == -1:
            kolory.append('red')
        else:
            kolory.append('blue')

    return G, polozenie, kolory, ramka
def zadanie5rys(G, polozenie, kolory):
    plt.figure(figsize=(10, 8))
    nx.draw(G, pos=polozenie, node_color=kolory, with_labels=True, node_size=1000, edge_color='gray', font_size=8)
    plt.show()
    G.clear()

G, polozenie, kolory, ramka = zadanie5przet(root)
zadanie5rys(G, polozenie, kolory)
ramka.to_csv("zadanie5.txt", index=False)

Zadanie6

In [ ]:
def zadanie6(root):
	try:
		leki = root.findall('drug')
	except:
		return None, -1
	sciezki = {}

	grupa_sciezek = root.findall('.//pathways')
	for gs in grupa_sciezek:
		for g in gs.findall('pathway'):
			id = g.findall('.//drugbank-id')
			for i in id:
				i = i.text
				if sciezki.get(i) is None:
					sciezki[i] = set()	
				sciezki[i].add(g.find('name').text)
				

	odp = []
	maks = 0
	ramka = []
	for x in sciezki.keys():
		akt = len(sciezki[x])
		if akt > maks:
			maks = akt
		ramka.append({'id' : x, 'ile' : akt})
		odp.append(akt)

	for l in leki:
		id = l.find("drugbank-id[@primary='true']").text
		if sciezki.get(id) is None:
			ramka.append({'id' : id, 'ile' : 0})
			odp.append(0)
	return odp, maks, pd.DataFrame(ramka)

odp, maks, ramka6 = zadanie6(root)
ramka6.to_csv('zadanie6.txt', index = False)
plt.hist(odp, bins=[x for x in range(maks + 2)], align='left')
plt.title("Histogram - z iloma szlakami wchodzą leki z bazy")
plt.xlabel("Z iloma szlakami")
plt.xticks([x for x in range(maks + 1)])
plt.ylabel("Ile leków")
plt.show()

Zadanie7


In [9]:
def zadanie7(root):
    try:
        leki = root.findall('.//pathways')
    except:
        return None
    ramka = []
    leki = root.findall('drug')
    
    for l in leki:
        id_leku = l.find("drugbank-id[@primary='true']").text

        for t in l.find('targets').findall('target'):
            id_bialka = t.find('id').text
            polipeptyd = t.find('polypeptide')

            if polipeptyd is None:
                ramka.append({
                    'id_leku': id_leku,
                    'id_bialka': id_bialka,
                    'zewnetrzna_baza': '-',
                    'id_w_zewnetrznej_bazie': '-',
                    'nazwa_polipeptydu': '-',
                    'gen_kodujacy': '-',
                    'GenAtlas_ID': '-',
                    'numer_chromosomu': '-',
                    'umiejscowienie_w_komorce': '-'
                })
            else:
                genatlas_id = '-'
                for r in polipeptyd.findall('external-identifier'):
                    if r.find('resource').text == 'GenAtlas':
                        genatlas_id = r.find('identifier').text
                        break
                ramka.append({
                    'id_leku': id_leku,
                    'id_bialka': id_bialka,
                    'zewnetrzna_baza': polipeptyd.get('source'),
                    'id_w_zewnetrznej_bazie': polipeptyd.get('id'),
                    'nazwa_polipeptydu': polipeptyd.find('name').text,
                    'gen_kodujacy': polipeptyd.find('gene-name').text,
                    'GenAtlas_ID': genatlas_id,
                    'numer_chromosomu': polipeptyd.find('chromosome-location').text,
                    'umiejscowienie_w_komorce': polipeptyd.find('cellular-location').text
                })
    
    return pd.DataFrame(ramka)
ramka = zadanie7(root)
ramka.to_csv("zadanie7.txt", index = False)

Zadanie8

In [ ]:
def zadanie8(root):
	odp = {}
	#korzystamy z ramki z zadania7
	ramka = zadanie7(root)
	if ramka is None:
		return
	for _, wiersz in zadanie7(root).iterrows():
		miejsce = wiersz['umiejscowienie_w_komorce']

		if miejsce != '-' and miejsce != '':
			if odp.get(miejsce) == None:
				odp[miejsce] = 0
			odp[miejsce] += 1

	plt.figure(figsize=(20, 10))
	kawalki, _ = plt.pie(odp.values())

	koncowa_legenda = []
	suma = 0
	for o in odp.values():
		suma += o
	for o in odp.keys():
		koncowa_legenda.append(f"{o} {round((odp[o]/suma)*100, 2)}%")

	plt.legend(kawalki, koncowa_legenda, title = 'Miejsca w komórce', loc = (-0.86, 0.6))
	plt.show()
zadanie8(root)

Zadanie9

In [ ]:
def zadanie9(root):
    grupy = {'Zaakceptowane': 0, 'Wycofane': 0, 'Eksperymentalne': 0, 'Leczenie_zwierzat': 0}
    try:
        leki = root.findall('drug')
    except:
        return None, None, None
    zatw_nie_wyc = 0
    
    for l in leki:
        akt_g = ''
        czy_z = 0
        for g in (l.find('groups')).findall('group'):
            akt_g = g.text
            if czy_z == 0 and akt_g == 'approved':
                czy_z = 1
            if czy_z == 1 and akt_g == 'withdrawn':
                czy_z = 2

        if (akt_g == 'approved'):
            grupy['Zaakceptowane'] += 1
        elif (akt_g == 'withdrawn'):
            grupy['Wycofane'] += 1
        elif (akt_g == 'experimental' or akt_g == 'investigational'):
            grupy['Eksperymentalne'] += 1
        elif (akt_g == 'vet_approved'):
            grupy['Leczenie_zwierzat'] += 1

        if czy_z == 2:
            zatw_nie_wyc += 1

    return pd.DataFrame([grupy]), grupy, zatw_nie_wyc

ramka, grupy, zatw_nie_wyc = zadanie9(root)
etykiety = []

for x in grupy.keys():
	etykiety.append(f"{x} - {grupy[x]}")


plt.pie(grupy.values(), labels = etykiety)
plt.show()
print('Zadanie9 - odpowiedź = ', zatw_nie_wyc)
ramka.to_csv("zadanie9.txt", index = False)

Zadanie10

In [12]:
def zadanie10(root):
	ramka = []
	try:
		leki = root.findall('drug')
	except:
		return None
	for l in leki:
		id_leku = l.find('drugbank-id', {'primary' : 'true'}).text
		for li in (l.find('drug-interactions')).findall('drug-interaction'):
			id_interakcji = li.find('drugbank-id').text
			interakcja = li.find('description').text
			ramka.append({
				'id_leku' : id_leku,
				'id_leku_wchodzacego_w_interakcje' : id_interakcji,
				'opis_interakcji' : interakcja
			})
	return pd.DataFrame(ramka)
ramka = zadanie10(root)
ramka.to_csv("Zadanie10.txt", index = False)

Zadanie11

In [ ]:
def zadanie11(gene_name):
	dane = []
	try:
		leki = root.findall('drug')
	except:
		return
	#Tworzymy graf, gdzie laczymy nazwe genu z lekiem, a lek z produktem
	G = nx.Graph()
	G.add_node(gene_name, color = 'purple')
	for l in leki:
		#na start szukamy dla kazdego leku gen i produkty. Zapisujemy je,
		#aby nastepnie wykorzystac do stworzenia prezentacji wizualnej.
		id_leku = l.find('drugbank-id', {'primary' : 'true'}).text
		nazwa_leku = l.find('name').text
		gen = 0

		for t in l.find('targets').findall('target'):
			polipeptydy = t.findall('polypeptide')
			for pol in polipeptydy:
				nazwa_genu = pol.find('gene-name').text
				if (nazwa_genu == gene_name):
					gen += 1

		produkty = []
		for p in l.find('products').findall('product'):
			x = p.find('name').text
			produkty.append(x)

		if gen != 0:
			G.add_node(f"{id_leku} {nazwa_leku}", color='red')
			G.add_edge(f"{id_leku} {nazwa_leku}", gene_name, color = 'green')
			for p in produkty:
				G.add_edge(f"{id_leku} {nazwa_leku}", p, color = 'gray')

	#Wyciagamy wczesciej ustalone kolory krawedzi, sa one trzecim parametrem w wierzcholku, zaraz
	#po wartosciach tych wierzcholkow.
	kolorye = [k["color"] for _, _, k in G.edges(data=True)]
	koloryw = [w["color"] if "color" in w else "blue" for _, w in G.nodes(data=True)]

	plt.figure(figsize=(15, 10))
	plt.tight_layout()
	nx.draw(G, with_labels = True, node_size = 500, font_size = 10, edge_color = kolorye, node_color = koloryw)
	plt.show()
	G.clear()

zadanie11("F2")

Zadanie12 - może coś dodać

In [ ]:
def zadanie12(root):
	try:
		leki = root.findall('drug')
	except:
		return None
	id_sciezek = set()
	sciezki_drugbank = []
	sciezki_smpdb = []
	for l in leki:
		for p in l.find('pathways').findall('pathway'):
			id = p.find('smpdb-id').text
			if id not in id_sciezek:
				id_sciezek.add(id)
				sciezki_drugbank.append({
					'id' : id,
					'nazwa' : p.find('name').text,
					'liczba_enzymow' : len(list(e for e in p.find('enzymes').findall('uniprot-id'))),
					'liczba_lekow' : len(list(lek for lek in p.find('drugs').findall('drug')))
				})

	wartosci = []
	nazwy = []
	with open('smpdb_pathways.csv', 'r', encoding = 'utf-8') as plik:
		reader = csv.reader(plik)
		for row in reader:
			id = row[0]
			if id in id_sciezek:
				for wiersz in sciezki_drugbank:
					if wiersz['id'] == id:
						wartosci.append(wiersz['liczba_enzymow'])
						nazwy.append(wiersz['nazwa'])
						sciezki_smpdb.append({
							'id' : id,
							'nazwa' : wiersz['nazwa'],
							'rodzaj' : row[3],
							'opis' : row[4],
							'liczba_enzymow' : wiersz['liczba_enzymow'],
							'liczba_lekow' : wiersz['liczba_lekow']
						})
						break

	plt.bar(nazwy, wartosci)
	plt.title('Wykres przedstawiający dla każdej ścieżki ile zawiera enzymów.')
	plt.xlabel('Nazwy ścieżek')
	plt.ylabel('Ile enzymów')
	plt.xticks(rotation=45, fontsize = 9)
	plt.subplots_adjust(bottom = 0.5)
	plt.show()

	return pd.DataFrame(sciezki_smpdb)
ramka = zadanie12(root)
ramka.to_csv("Zadanie12paths.txt", index = False)

#szukamy dla kazdej bazy, ilu polipeptydow id trzyma
def zadanie12b(root):
	try:
		pol = root.findall('.//polypeptide')
	except:
		return None
	id_resources = {}

	for p in pol:
		id = p.get('id')
		for r in p.findall('.//external-identifier'):
			nazwa = r.find('resource').text
			if id_resources.get(nazwa) is None:
				id_resources[nazwa] = set()
			id_resources[nazwa].add(id)

	nazwy = []
	wartosci = []
	ramka = []
	for i, j in id_resources.items():
		if i is None:
			continue
		nazwy.append(i)
		wartosci.append(len(j))
		ramka.append({'nazwa_bazy' : i, 'ile' : len(j)})
	
	plt.figure(figsize = (20, 14))
	plt.bar(nazwy, wartosci)
	plt.title('Wykres przedstawiający dla każdej ścieżki ile zawiera enzymów.')
	plt.xlabel('Nazwy ścieżek')
	plt.ylabel('Ile enzymów')
	plt.xticks(rotation=45, fontsize = 9)
	plt.subplots_adjust(bottom = 0.5)
	plt.show()

	return pd.DataFrame(ramka)

ramka = zadanie12b(root)
ramka.to_csv("zadanie12b.txt", index = False)

Zadanie13

In [ ]:
def zadanie13(root, ile_gen, nazwa_pliku):
    if ile_gen < 0:
        return 1
    try:
        leki = root.findall("drug")
    except:
        return 2
    def daty():
        #generuje przykladowe daty, wybor lat jest losowy, mozna zmienic przedzialy
        data1 = datetime(random.randint(2000, 2015), random.randint(1, 12), random.randint(1,28))
        data2 = datetime(random.randint(data1.year+1, 2024), random.randint(1,12), random.randint(1,28))
        return data1.strftime('%Y-%m-%d'), data2.strftime('%Y-%m-%d')


    typy = []
    for drug in leki:
        drug_type = drug.attrib.get('type')
        if drug_type:
            typy.append(drug_type)

    oryginalne = {}
    for drug in root.findall("drug"):
        for child in drug:
            if child.tag not in oryginalne:
                oryginalne[child.tag] = []
            oryginalne[child.tag].append(child)

    max_id = 0
    for drug in root.findall("drug"):
        drug_id = drug.find("drugbank-id[@primary='true']").text
        current_id = int(drug_id.lstrip("DB"))
        max_id = max(max_id, current_id)

    def plik(root, typy, oryginalne, max_id, ile_dodac, nazwa_pliku):
        with open(nazwa_pliku, "w", encoding="utf-8") as plik:
            plik.write(etree.tostring(root, pretty_print=True, encoding='unicode'))

        with open(nazwa_pliku, "r", encoding="utf-8") as plik:
            lines = plik.readlines()

        with open(nazwa_pliku, "w", encoding="utf-8") as plik:
            plik.writelines(lines[:-1])
            #generujemy 2k, bo po wygenerowaniu 20k dane sa za duze, i niezaleznie czy z uzyciem soup czy lxml, moj komputer nie
            #wytrzymuje i po chwili cały ram jest zajety
            for i in range(ile_dodac):
                d1, d2 = daty()
                new_drug = etree.Element('drug', xmlns="http://www.drugbank.ca", type = random.choice(typy), create = d1, updated = d2)
                
                max_id += 1
                new_id = f"DB{max_id:05d}"
                drugbank_id = etree.SubElement(new_drug, 'drugbank-id', primary='true')
                drugbank_id.text = new_id
                
                a = 0
                for k in oryginalne:
                    if a == 0:
                        a+=1
                        continue
                    el = random.choice(oryginalne[k])
                    new_drug.append(copy.deepcopy(el))

                plik.write(etree.tostring(new_drug, pretty_print=True, encoding='unicode')+"\n")

            plik.write("\n</drugbank>\n")

    plik(root, typy, oryginalne, max_id, ile_gen, nazwa_pliku)
    return 0
zadanie13(root, 1900, 'drugbank_gen.xml')

Zadanie14 - korzystamy z ipytest, aby móc puścić testy w notebooku

In [ ]:
ipytest.autoconfig()
#test sprawdza, czy dla testowych danych wczytane dane zawierają 100 lekow, z czego max id leku to DB00108
def test_input():
    root = odczyt("drugbank_partial.xml")
    ile_lekow = 0
    max_id = 0
    for l in root.findall('drug'):
        ile_lekow += 1
        current_id = int(l.find("drugbank-id[@primary='true']").text.lstrip("DB"))
        max_id = max(max_id, current_id)
    assert(ile_lekow == 100)
    assert(max_id == 108)

def test_zadania_dane_partial():
    root = odczyt("drugbank_partial.xml")

    #zadanie1
    ramka1 = zadanie1(root)
    assert(len(ramka1) == 100)

    #zadanie2
    ramka2 = zadanie2przet(root)
    maks_synonimy = 0
    min_synonimy = 5000
    for _, wiersz in ramka2.iterrows():
        maks_synonimy = max(maks_synonimy, len(wiersz['synonimy']))
        min_synonimy = min(min_synonimy, len(wiersz['synonimy']))
    assert(maks_synonimy == 31)
    assert(min_synonimy == 1)

    #zadanie3
    ramka3 = zadanie3(root)
    ile = 0
    ile_nie_none = 0
    curr_id = 0
    for _, wiersz in ramka3.iterrows():
        if wiersz['kod_ndc'] != None and curr_id != wiersz['id']:
            ile_nie_none += 1
            curr_id = wiersz['id']
        ile += 1
    assert(ile_nie_none == 88)
    assert(ile == len(ramka3) and ile == 4584)

    #zadanie5
    G, _, _, _ = zadanie5przet(root)
    assert(G.number_of_nodes() == 26)
    assert(G.number_of_edges() == 28)

    #zadanie7
    ramka7 = zadanie7(root)
    #sprawdzamy ile nie ma polipeptydow
    ile = 0
    for _, wiersz in ramka7.iterrows():
        i = 0
        for w in wiersz:
            if i < 2:
                i += 1
                continue
            if w != '-':
                i = 3
                break
        if i != 3:
            ile += 1

    assert(ile == 18)
    assert(len(ramka7) == 267)

    #zadanie9
    _, grupy, _ = zadanie9(root)
    assert(grupy['Zaakceptowane'] == 37 and grupy['Eksperymentalne'] == 49 and grupy['Leczenie_zwierzat'] == 4 and grupy['Wycofane'] == 10)

    #zadanie10
    ramka10 = zadanie10(root)
    assert(len(ramka10) == 50688)
    
def test_mocking_pathways():
    dane = """
    <drugbank>
        <pathways>
            <pathway>
                <smpdb-id>path1</smpdb-id>
                <name>Pathway 1</name>
                <drugs>
                    <drug>
                        <drugbank-id>drug1</drugbank-id>
                        <name>Drug A</name>
                    </drug>
                    <drug>
                        <drugbank-id>drug2</drugbank-id>
                        <name>Drug B</name>
                    </drug>
                </drugs>
            </pathway>
            <pathway>
                <smpdb-id>path2</smpdb-id>
                <name>Pathway 2</name>
                <drugs>
                    <drug>
                        <drugbank-id>drug3</drugbank-id>
                        <name>Drug C</name>
                    </drug>
                </drugs>
            </pathway>
        </pathways>
        <pathways>
            <pathway>
                <smpdb-id>path3</smpdb-id>
                <name>Pathway 3</name>
                <drugs>
                    <drug>
                        <drugbank-id>drug1</drugbank-id>
                        <name>Drug A</name>
                    </drug>
                </drugs>
            </pathway>
        </pathways>
    </drugbank>
    """
    root = etree.fromstring(dane)
    G, _, _, _ = zadanie5przet(root)
    assert(G.number_of_edges() == 4 and G.number_of_nodes() == 6)

def test_parametryzacja_generowanie():
    root_start = odczyt('drugbank_partial.xml')
    zadanie13(root_start, 100, 'drugbank_gen.xml')
    root = odczyt('drugbank_gen.xml')
    assert(len(root.findall('drug')) == 200)
    zadanie13(root_start, 50, 'drugbank_gen.xml')
    root = odczyt('drugbank_gen.xml')
    assert(len(root.findall('drug')) == 150)
    ret = zadanie13(root_start, -10, 'drugbank_gen.xml')
    assert(ret == 1)
    zadanie13(root_start, 400, 'drugbank_gen.xml')
    root = odczyt('drugbank_gen.xml')
    assert(len(root.findall('drug')) == 500)

def test_mocking_targets():
    dane = """
    <drugbank>
        <drug>
            <drugbank-id primary="true">DB0001</drugbank-id>
            <targets>
                <target>
                    <id>TARGET001</id>
                    <polypeptide source="GenAtlas" id="PEP001">
                        <name>Polypeptide A</name>
                        <gene-name>GENA</gene-name>
                        <chromosome-location>1p36</chromosome-location>
                        <cellular-location>Membrane</cellular-location>
                        <external-identifier>
                            <resource>GenAtlas</resource>
                            <identifier>GA001</identifier>
                        </external-identifier>
                    </polypeptide>
                </target>
            </targets>
        </drug>
        <drug>
            <drugbank-id primary="true">DB0002</drugbank-id>
            <targets>
                <target>
                    <id>TARGET002</id>
                    <polypeptide source="GenAtlas" id="PEP002">
                        <name>Polypeptide B</name>
                        <gene-name>GENB</gene-name>
                        <chromosome-location>2q21</chromosome-location>
                        <cellular-location>Cytoplasm</cellular-location>
                        <external-identifier>
                            <resource>GenAtlas</resource>
                            <identifier>GA002</identifier>
                        </external-identifier>
                    </polypeptide>
                </target>
            </targets>
        </drug>
        <drug>
            <drugbank-id primary="true">DB0003</drugbank-id>
            <targets>
                <target>
                    <id>TARGET003</id>
                </target>
            </targets>
        </drug>
    </drugbank>
    """
    ramka = zadanie7(etree.fromstring(dane))
    assert(len(ramka) == 3)
    ile = 0
    for _, wiersz in ramka.iterrows():
        if wiersz['zewnetrzna_baza'] == '-':
            ile += 1
    assert(ile == 1)

def test_zle_nazwy_pliku():
    root1 = odczyt('dd.xml')
    assert(root1 is None)
    root2 = odczyt('drugbank_partial.txt')
    assert(root2 is None)
    root3 = odczyt('drugbank_partial.xml')
    assert(root3 is not None)

def test_zle_dane_funkcji():
    ret = zadanie1(None)
    assert(ret == None)
    ret = zadanie2przet(None)
    assert(ret == None)
    ret = zadanie3(None)
    assert(ret == None)
    ret = zadanie4(None)
    assert(ret == -1)
    ret, _, _, _ = zadanie5przet(None)
    assert(ret == None)
    ret, _ = zadanie6(None)
    assert(ret == None)
    ret = zadanie7(None)
    assert(ret == None)
    ret, _, _ = zadanie9(None)
    assert(ret == None)
    ret = zadanie10(None)
    assert(ret == None)
    ret = zadanie13(None, 1, 'a')
    assert(ret == 2)

def test_mocking_niepoprawny_xml():
    dane = """
    <drugbank>
        <drug>
            <drugbank-id primary="true">DB001</drugbank-id>
            <name>Drug A</name>
            <targets>
                <target>
                    <id>12345</id>
                    <polypeptide>
                        <name>Polypeptide A</name>
                        <gene-name>Gene A</gene-name>
                        <chromosome-location>12q21</chromosome-location>
                        <cellular-location>Membrane</cellular-location>
                    </polypeptide>
                </target>
            </targets>
        </drug>

        <drug>
            <drugbank-id primary="true">DB002</drugbank-id>
            <name>Drug B</name>
            <targets>
                <target>
                    <id>67890</id>
                </target>
            </targets>
    </drugbank>
    """
    #W kazdym zadaniu tak samo sprawdzamy poprawnosc danych
    ret = zadanie1(dane)
    assert(ret == None)
    with open('pytest.xml', 'w', encoding = 'utf-8') as plik:
        plik.write(dane)
    ret = odczyt('pytest.xml')
    assert(ret == None)
    os.remove('pytest.xml')

def test_poprawnosc_kolorowania_graf5():
    root = odczyt('drugbank_partial.xml')
    G, polozenie, kolory, _ = zadanie5przet(root)
    for g, kolor in zip(G.nodes(), kolory):
        assert ((polozenie[g][0] == -1 and kolor == 'red') or (polozenie[g][0] == 1 and kolor == 'blue'))

def test_mocking_zadanie4():
    dane = """
    <drugbank>
        <pathways>
            <pathway>
                <smpdb-id>path1</smpdb-id>
                <name>Pathway 1</name>
                <drugs>
                    <drug>
                        <drugbank-id>DB0001</drugbank-id>
                        <name>Drug A</name>
                    </drug>
                </drugs>
            </pathway>
            <pathway>
                <smpdb-id>path2</smpdb-id>
                <name>Pathway 2</name>
                <drugs>
                    <drug>
                        <drugbank-id>DB0002</drugbank-id>
                        <name>Drug B</name>
                    </drug>
                </drugs>
            </pathway>
        </pathways>
        <pathways>
            <pathway>
                <smpdb-id>path3</smpdb-id>
                <name>Pathway 3</name>
                <drugs>
                    <drug>
                        <drugbank-id>DB0003</drugbank-id>
                        <name>Drug C</name>
                    </drug>
                </drugs>
            </pathway>
        </pathways>
    </drugbank>
    """
    root = etree.fromstring(dane)
    assert (zadanie4(root) == 3)

    

ipytest.run()

Zadanie15

In [ ]:
from pydantic import BaseModel
from fastapi import FastAPI, HTTPException
import nest_asyncio
import uvicorn
import requests


nest_asyncio.apply()

#krotsza wersja, zwracajaca tylko slownik id: set sciezek
def zadanie6uciete(root):
    try:
        leki = root.findall('drug')
    except:
        return None, -1
    sciezki = {}
    for l in leki:
            id = l.find("drugbank-id[@primary='true']").text
            sciezki[id] = set()

    grupa_sciezek = root.findall('.//pathways')
    for gs in grupa_sciezek:
        for g in gs.findall('pathway'):
            id = g.findall('.//drugbank-id')
            for i in id:
                i = i.text
                if sciezki.get(i) is None:
                    sciezki[i] = set()	
                sciezki[i].add(g.find('name').text)
                

    return sciezki

#towrzymy slownik przed, gdybysmy chcieli wywolywac czesciej, bo optymalniejsze
sciezki = zadanie6uciete(root)
app = FastAPI()

# Model do walidacji ID leku
class DrugID(BaseModel):
    drug_id: str

# Endpoint POST do obliczania liczby szlaków
@app.post("/drug_interaction/")
async def drug_interaction(drug: DrugID):
    drug_id = drug.drug_id
    if sciezki.get(drug_id) is None:
        raise HTTPException(status_code=404, detail="Nie znaleziono leku")
    else:
        return {"drug_id": drug_id, "liczba_interakcji": len(sciezki[drug_id])}
		


# Funkcja uruchamiająca serwer FastAPI
def run_app():
    uvicorn.run(app, host="127.0.0.1", port=8000)

# Uruchamiamy serwer w osobnym wątku
thread = Thread(target=run_app)
#zakonczy sie gdy zamkniemy notebook
thread.daemon = True
thread.start()

try:
    response = requests.post(
        "http://127.0.0.1:8000/drug_interaction/",
        json={"drug_id": "DB00102"}  # Przykładowe ID
    )
    print(f"{response.json()}")
except Exception as e:
    print(f"Błąd: {e}")
